# GridLock Hackathon 2.0 — v9 (LightGBM + Lookup + Blends, self-contained)

Single notebook produces:  
1. `submission_lgbm.csv`     — v3-style LightGBM with OOF encodings (target: ~90)  
2. `submission_nearest.csv`  — pure day-48 nearest lookup (proven: 79.55)  
3. `submission_blend_a*.csv` — alpha-blends at α ∈ {0.5..0.95}  
4. `submission_blend_smart.csv` — per-row disagreement-weighted blend  

No external dependencies, no missing-file requirements. Run top to bottom.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from bisect import bisect_left
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stop, log_evaluation as lgb_log
import pygeohash as pgh

pd.set_option('display.max_columns', 60)
np.random.seed(42)
print('Imports OK')

## Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

for df in [train, test]:
    df['hour']    = df['timestamp'].str.split(':').str[0].astype(int)
    df['minute']  = df['timestamp'].str.split(':').str[1].astype(int)
    df['ts_mins'] = df['hour'] * 60 + df['minute']

print(f'Train: {train.shape}  |  Test: {test.shape}')

---
## Step A: v3-Style Feature Engineering

In [ ]:
class FeatureEngineer:
    @staticmethod
    def _decode_gh(gh):
        try:
            lat, lng, lat_err, lng_err = pgh.decode_exactly(gh)
            return float(lat), float(lng), float(lat_err), float(lng_err)
        except Exception:
            return 0.0, 0.0, 0.0, 0.0

    @staticmethod
    def _time_of_day(h):
        if   h <= 5:  return 0
        elif h <= 11: return 1
        elif h <= 17: return 2
        else:         return 3

    def fit_transform(self, df):
        df = df.copy()
        self._temp_med_global = float(df['Temperature'].median())
        self._temp_med_gh     = df.groupby('geohash')['Temperature'].median()
        self._rt_mode_global  = (df['RoadType'].mode().iloc[0]
                                 if len(df['RoadType'].mode()) else 'Residential')
        self._rt_mode_gh      = df.groupby('geohash')['RoadType'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._rt_mode_global)
        self._wx_mode_global  = (df['Weather'].mode().iloc[0]
                                 if len(df['Weather'].mode()) else 'Sunny')
        self._wx_mode_gh      = df.groupby('geohash')['Weather'].apply(
            lambda x: x.mode().iloc[0] if len(x.mode()) else self._wx_mode_global)

        df = self._impute(df)

        self._max_day = int(df['day'].max())
        self._rt_cols = sorted(df['RoadType'].dropna().unique().tolist())
        self._wx_cols = sorted(df['Weather'].dropna().unique().tolist())

        gb = df.groupby('geohash')['demand']
        self._gh_std   = gb.std().fillna(0)
        self._gh_p25   = gb.quantile(0.25)
        self._gh_p75   = gb.quantile(0.75)
        self._gh_max   = gb.max()
        self._gh_count = gb.count().astype(float)
        self._gh_mean  = gb.mean()
        self._g_mean   = float(df['demand'].mean())
        self._g_std    = float(df['demand'].std())

        self._hour_mean = df.groupby('hour')['demand'].mean()
        self._hour_std  = df.groupby('hour')['demand'].std().fillna(0)
        tmp_tod = df['hour'].apply(self._time_of_day)
        df['_tod'] = tmp_tod
        self._tod_mean = df.groupby('_tod')['demand'].mean()
        df = df.drop(columns=['_tod'])

        gh_mean_d = self._gh_mean.to_dict()
        self._neighbor_mean_d = {}
        for gh in df['geohash'].unique():
            try:
                nbrs = list(pgh.neighbors(gh).values())
                self._neighbor_mean_d[gh] = float(np.mean(
                    [gh_mean_d.get(n, self._g_mean) for n in nbrs]))
            except Exception:
                self._neighbor_mean_d[gh] = self._g_mean

        gh4 = df['geohash'].str[:4]
        gh5 = df['geohash'].str[:5]
        self._gh4_labels = {v: i for i, v in enumerate(sorted(gh4.unique()))}
        self._gh5_labels = {v: i for i, v in enumerate(sorted(gh5.unique()))}

        _, self._temp_bin_edges = pd.cut(df['Temperature'], bins=5, retbins=True)
        self._temp_bin_edges[0]  = -np.inf
        self._temp_bin_edges[-1] =  np.inf

        self._gh_rank = self._gh_mean.rank(pct=True).to_dict()

        self.fitted = True
        return self._transform(df)

    def transform(self, df):
        assert self.fitted
        df = df.copy()
        df = self._impute(df)
        return self._transform(df)

    def _impute(self, df):
        df['Temperature'] = df['Temperature'].fillna(
            df['geohash'].map(self._temp_med_gh).fillna(self._temp_med_global))
        df['RoadType'] = df['RoadType'].fillna(
            df['geohash'].map(self._rt_mode_gh).fillna(self._rt_mode_global))
        df['Weather'] = df['Weather'].fillna(
            df['geohash'].map(self._wx_mode_gh).fillna(self._wx_mode_global))
        return df

    def _transform(self, df):
        df['hour_sin']     = np.sin(2*np.pi*df['hour']/24)
        df['hour_cos']     = np.cos(2*np.pi*df['hour']/24)
        df['minute_sin']   = np.sin(2*np.pi*df['minute']/60)
        df['minute_cos']   = np.cos(2*np.pi*df['minute']/60)
        df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
        df['time_of_day']  = df['hour'].apply(self._time_of_day)

        geo = df['geohash'].apply(self._decode_gh)
        df['geohash_lat']     = geo.apply(lambda x: x[0])
        df['geohash_lng']     = geo.apply(lambda x: x[1])
        df['geohash_lat_err'] = geo.apply(lambda x: x[2])
        df['geohash_lng_err'] = geo.apply(lambda x: x[3])

        df['day_sin']   = np.sin(2*np.pi*df['day']/self._max_day)
        df['day_cos']   = np.cos(2*np.pi*df['day']/self._max_day)
        df['day_mod_7'] = df['day'] % 7

        df['LargeVehicles'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks']     = (df['Landmarks']     == 'Yes').astype(int)
        for rt in self._rt_cols:
            df[f'RoadType_{rt}'] = (df['RoadType'] == rt).astype(int)
        for wx in self._wx_cols:
            df[f'Weather_{wx}']  = (df['Weather']  == wx).astype(int)

        df['geohash_std_demand'] = df['geohash'].map(self._gh_std).fillna(self._g_std)
        df['geohash_p25_demand'] = df['geohash'].map(self._gh_p25).fillna(self._g_mean*0.4)
        df['geohash_p75_demand'] = df['geohash'].map(self._gh_p75).fillna(self._g_mean*1.6)
        df['geohash_max_demand'] = df['geohash'].map(self._gh_max).fillna(self._g_mean*2.5)
        df['geohash_count']      = df['geohash'].map(self._gh_count).fillna(1.0)
        df['neighbor_mean_demand'] = df['geohash'].map(self._neighbor_mean_d).fillna(self._g_mean)

        df['hour_mean_demand'] = df['hour'].map(self._hour_mean).fillna(self._g_mean)
        df['hour_std_demand']  = df['hour'].map(self._hour_std ).fillna(self._g_std)
        df['tod_mean_demand']  = df['time_of_day'].map(self._tod_mean).fillna(self._g_mean)

        df['geohash_4'] = df['geohash'].str[:4]
        df['geohash_5'] = df['geohash'].str[:5]
        df['geohash_4_label'] = df['geohash_4'].map(self._gh4_labels).fillna(-1).astype(int)
        df['geohash_5_label'] = df['geohash_5'].map(self._gh5_labels).fillna(-1).astype(int)

        df['temp_binned'] = pd.cut(df['Temperature'], bins=self._temp_bin_edges,
                                     labels=False, include_lowest=True).fillna(2).astype(int)
        df['demand_rank_in_geohash'] = df['geohash'].map(self._gh_rank).fillna(0.5)

        rt_num = {'Highway': 3, 'Street': 2, 'Residential': 1}
        df['RoadType_numeric'] = df['RoadType'].map(rt_num).fillna(0).astype(float)
        df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType_numeric']

        wx_num = {'Sunny': 1, 'Cloudy': 2, 'Rainy': 3, 'Foggy': 4, 'Snowy': 5}
        df['Weather_numeric'] = df['Weather'].map(wx_num).fillna(1).astype(float)
        df['temp_x_weather']  = df['Temperature'] * df['Weather_numeric']

        df['hour_x_geohash_lat'] = df['hour'] * df['geohash_lat']
        return df

fe = FeatureEngineer()
train_fe = fe.fit_transform(train)
test_fe  = fe.transform(test)
print(f'Train FE: {train_fe.shape}  |  Test FE: {test_fe.shape}')

---
## Step B: OOF Target Encoding (α=10, time-sorted KFold)

In [ ]:
def oof_encode(train_df, test_df, group_cols, col_name,
               target='demand', n_splits=5, alpha=10):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    gm = float(train_df[target].mean())
    n  = len(train_df)

    sort_key   = (train_df['day'] * 1440 + train_df['hour'] * 60 + train_df['minute']).values
    sort_pos   = np.argsort(sort_key)
    unsort_pos = np.argsort(sort_pos)
    df_s = train_df.iloc[sort_pos].reset_index(drop=True)

    enc_sorted = np.full(n, gm, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=False)

    for tr_idx, va_idx in kf.split(np.arange(n)):
        stats = df_s.iloc[tr_idx].groupby(group_cols)[target].agg(['mean', 'count'])
        stats['smooth'] = (stats['count']*stats['mean'] + alpha*gm) / (stats['count'] + alpha)
        d = stats['smooth'].to_dict()
        if len(group_cols) == 1:
            keys = df_s.iloc[va_idx][group_cols[0]].tolist()
        else:
            keys = list(zip(*[df_s.iloc[va_idx][c].tolist() for c in group_cols]))
        enc_sorted[va_idx] = [d.get(k, gm) for k in keys]

    train_df[col_name] = enc_sorted[unsort_pos]

    full = train_df.groupby(group_cols)[target].agg(['mean', 'count'])
    full['smooth'] = (full['count']*full['mean'] + alpha*gm) / (full['count'] + alpha)
    full_d = full['smooth'].to_dict()
    if len(group_cols) == 1:
        te_keys = test_df[group_cols[0]].tolist()
    else:
        te_keys = list(zip(*[test_df[c].tolist() for c in group_cols]))
    test_df[col_name] = [full_d.get(k, gm) for k in te_keys]

print('Applying OOF encodings...')
oof_encode(train_fe, test_fe, 'geohash',                  'geohash_mean_demand');           print('  1/6')
oof_encode(train_fe, test_fe, ['geohash', 'hour'],         'geohash_hour_mean_demand');      print('  2/6')
oof_encode(train_fe, test_fe, ['geohash', 'time_of_day'],  'geohash_timeofday_mean_demand'); print('  3/6')
oof_encode(train_fe, test_fe, 'geohash_4',                 'geohash_4_mean_demand');         print('  4/6')
oof_encode(train_fe, test_fe, 'geohash_5',                 'geohash_5_mean_demand');         print('  5/6')
oof_encode(train_fe, test_fe, 'temp_binned',               'temp_binned_mean_demand');       print('  6/6')

# Post-OOF derived
for df in [train_fe, test_fe]:
    df['location_time']     = df['geohash_mean_demand']      * df['hour_sin']
    df['demand_vs_hour']    = df['geohash_mean_demand']      / (df['hour_mean_demand'] + 1e-9)
    df['gh_hour_vs_global'] = df['geohash_hour_mean_demand'] / (df['hour_mean_demand'] + 1e-9)

print(f'Done. Train cols: {len(train_fe.columns)}, Test cols: {len(test_fe.columns)}')

---
## Step C: Train LightGBM

Fixed good hyperparams (no Optuna to save time). Time-aware train/val split (last 15%) for early-stopping reference, then retrain on full train.

In [ ]:
EXCLUDE = {'Index', 'geohash', 'timestamp', 'demand', 'RoadType', 'Weather',
           'geohash_4', 'geohash_5'}
feature_cols = [c for c in train_fe.columns if c not in EXCLUDE]
print(f'Features: {len(feature_cols)}')

X_all = train_fe[feature_cols].values.astype(np.float32)
y_all = train_fe['demand'].values.astype(np.float32)
X_test_lgbm = test_fe[feature_cols].values.astype(np.float32)

for arr in [X_all, X_test_lgbm]:
    if np.isnan(arr).any():
        arr[:] = np.nan_to_num(arr, nan=0.0)

sort_keys  = (train_fe['day']*1440 + train_fe['hour']*60 + train_fe['minute']).values
sorted_idx = np.argsort(sort_keys)
split = int(len(sorted_idx) * 0.85)
tr_idx = sorted_idx[:split]; va_idx = sorted_idx[split:]
X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_va, y_va = X_all[va_idx], y_all[va_idx]

PARAMS = dict(
    learning_rate=0.05, num_leaves=255, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    n_estimators=3000, random_state=42, n_jobs=-1, verbose=-1
)

print('\nTraining LightGBM with early-stopping on last-15% time slice...')
val_model = LGBMRegressor(**PARAMS)
val_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
               callbacks=[lgb_early_stop(50, verbose=False), lgb_log(200)])

tr_r2 = r2_score(y_tr, val_model.predict(X_tr))
va_r2 = r2_score(y_va, val_model.predict(X_va))
best_iter = val_model.best_iteration_
print(f'\nTrain R^2={tr_r2:.4f}  Val R^2={va_r2:.4f}  Gap={tr_r2-va_r2:.4f}')
print(f'Best iter: {best_iter}')

# Retrain on FULL train with best_iter * 1.15
final_iters = int(best_iter * 1.15) + 1
final_params = {k: v for k, v in PARAMS.items() if k != 'n_estimators'}
final_params['n_estimators'] = final_iters
print(f'\nRetraining on full train ({X_all.shape[0]} rows, {final_iters} trees)...')
final_model = LGBMRegressor(**final_params)
final_model.fit(X_all, y_all)

preds_lgbm = final_model.predict(X_test_lgbm)
preds_lgbm = np.clip(preds_lgbm, 0.0, 1.0)
print(f'LightGBM preds: min={preds_lgbm.min():.4f} max={preds_lgbm.max():.4f} '
      f'mean={preds_lgbm.mean():.4f} std={preds_lgbm.std():.4f}')

# Save
sub_lgbm = pd.DataFrame({'Index': test['Index'].values, 'demand': preds_lgbm})
assert sub_lgbm.shape == (41778, 2)
sub_lgbm.to_csv('submission_lgbm.csv', index=False)
print('submission_lgbm.csv saved')

---
## Step D: Day-48 Nearest Lookup (pure dict/numpy)

In [ ]:
d48 = train[train['day'] == 48]
gh_arr_48 = d48['geohash'].values
ts_arr_48 = d48['ts_mins'].values
dm_arr_48 = d48['demand'].values

gh_exact = {}
gh_records_unsorted = {}
for i in range(len(d48)):
    g = gh_arr_48[i]; ts = int(ts_arr_48[i]); d = float(dm_arr_48[i])
    gh_exact[(g, ts)] = d
    gh_records_unsorted.setdefault(g, []).append((ts, d))
gh_records = {g: sorted(lst, key=lambda x: x[0]) for g, lst in gh_records_unsorted.items()}
gh_mean_full = train.groupby('geohash')['demand'].mean().to_dict()
global_mean = float(train['demand'].mean())

def nearest(g, ts):
    rec = gh_records[g]
    ts_list = [r[0] for r in rec]
    pos = bisect_left(ts_list, ts)
    if pos == 0: return rec[0][1]
    if pos == len(rec): return rec[-1][1]
    b, a = rec[pos-1], rec[pos]
    return b[1] if (ts - b[0]) <= (a[0] - ts) else a[1]

test_gh = test['geohash'].values
test_ts = test['ts_mins'].values.astype(int)
n_test  = len(test)

preds_lookup = np.empty(n_test, dtype=np.float64)
src = []
for i in range(n_test):
    g, ts = test_gh[i], int(test_ts[i])
    key = (g, ts)
    if key in gh_exact:
        preds_lookup[i] = gh_exact[key]; src.append('exact')
    elif g in gh_records:
        preds_lookup[i] = nearest(g, ts);  src.append('nearest')
    elif g in gh_mean_full:
        preds_lookup[i] = gh_mean_full[g]; src.append('gh_mean')
    else:
        preds_lookup[i] = global_mean;     src.append('global')

print('Source breakdown:', pd.Series(src).value_counts().to_dict())
print(f'Lookup preds: min={preds_lookup.min():.4f} max={preds_lookup.max():.4f} '
      f'mean={preds_lookup.mean():.4f} std={preds_lookup.std():.4f}')

sub_lookup = pd.DataFrame({'Index': test['Index'].values, 'demand': preds_lookup})
assert sub_lookup.shape == (41778, 2)
sub_lookup.to_csv('submission_nearest.csv', index=False)
print('submission_nearest.csv saved')

---
## Step E: Alpha-Blends + Smart Per-Row Blend

`blend = α · LightGBM + (1-α) · lookup`, clipped to [0, 1].  
Smart blend weights each row by disagreement between the two models.

In [ ]:
corr = np.corrcoef(preds_lgbm, preds_lookup)[0, 1]
print(f'Correlation between LightGBM and lookup: {corr:.4f}')
print(f'Mean abs diff: {np.abs(preds_lgbm - preds_lookup).mean():.4f}\n')

alphas = [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
print('Uniform alpha-blends:')
for a in alphas:
    p = a * preds_lgbm + (1 - a) * preds_lookup
    p = np.clip(p, 0.0, 1.0)
    fname = f'submission_blend_a{int(a*100):02d}.csv'
    pd.DataFrame({'Index': test['Index'].values, 'demand': p}).to_csv(fname, index=False)
    print(f'  α={a:.2f} → {fname}  '
          f'mean={p.mean():.4f} std={p.std():.4f}')

# Smart per-row blend
disagree = np.abs(preds_lgbm - preds_lookup)
scale = np.percentile(disagree, 90) + 1e-9
w_lgbm = 0.3 + 0.6 * np.clip(disagree / scale, 0.0, 1.0)
preds_smart = w_lgbm * preds_lgbm + (1 - w_lgbm) * preds_lookup
preds_smart = np.clip(preds_smart, 0.0, 1.0)
pd.DataFrame({'Index': test['Index'].values, 'demand': preds_smart}).to_csv(
    'submission_blend_smart.csv', index=False)
print(f'\nSmart blend → submission_blend_smart.csv  '
      f'mean={preds_smart.mean():.4f} std={preds_smart.std():.4f}')

print(f'\n=== SUBMISSION STRATEGY ===')
print('  1. submission_lgbm.csv         (pure LightGBM, expected ~88-90)')
print('  2. submission_nearest.csv      (pure lookup, proven 79.55)')
print('  3. submission_blend_a90.csv    (90% LightGBM + 10% lookup)')
print('  4. submission_blend_a80.csv    (80% LightGBM + 20% lookup)')
print('  5. submission_blend_a70.csv    (70% LightGBM + 30% lookup)')
print('  6. submission_blend_smart.csv  (disagreement-weighted)')
print('\nSubmit in this order — the alpha sweet spot usually lands in 0.7-0.9.')